# Install Dependencies

In [ ]:
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.3/366.3 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 26.1 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets
!pip install -q regex

In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 2.5 MB/s eta 0:00:00


In [ ]:
!pip install -q diffusers

# login

In [ ]:
import huggingface_hub
huggingface_hub.login('HF_TOKEN')

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.utils import shuffle
import os
import re
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             precision_score,
                             recall_score,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
import emoji
import pyarabic.araby as araby

In [ ]:
import pandas as pd

In [ ]:
import torch
import torch.distributed as dist

# Load Model

In [ ]:
model_name = "QCRI/Fanar-1-9B-Instruct"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )

# Assign pad_token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/900 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.69G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/2.12M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/18.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

# Load Data

In [ ]:
import pandas as pd
data = pd.read_excel('sample_biology.xlsx')

In [ ]:
data.head()

,ID,Source,Country,Group,Subject,Level,Question,Context,Answer Key,Option 1,Option 2,Option 3,Option 4,Option 5,is_few_shot
0,9998,بنك أسئلة الفصل الدراسي الأول-2.pdf - Google D...,Jordan,STEM,Biology,High,إذا كان عدد جزيئات مستقبل الالكترونات النهائي ...,NaN,D,24,18,12,36,NaN,0
1,10828,https://docs.google.com/forms/d/e/1FAIpQLSfOH9...,Egypt,STEM,Biology,High,تتحول الخلايا ..... إلى خلايا بلعمية فى بعض ال...,NaN,B,الخلايا التائية,الخلايا وحيدة النواة,الخلايا الحامضية,الخلايا القاتلة الطبيعية,NaN,0
2,11086,exams-28a8e986-7722-11ea-9116-54bef70b159e,NaN,STEM,Biology,High,ما اسم الوعاء الدموي الذي يحمل الدم المؤكسج بع...,NaN,B,الصمام,الشرايين,الشعيرات,الأوردة,NaN,0
3,9740,https://drive.google.com/file/d/1XyNWikrWZ01Gi...,Palestine,STEM,Biology,High,خلية تحتوي على 16 كروموسوم في المرحلة النمو ال...,NaN,B,8,16,32,35,NaN,0
4,10862,https://docs.google.com/forms/d/e/1FAIpQLScuuw...,Egypt,STEM,Biology,High,كل المواد التالية وسائل مناعة تركيبية موجودة س...,NaN,D,شموع,شعيرات,اشواك,اصماغ,NaN,0


# Zero Shot

In [ ]:
content = '''لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد الإجابة الصحيحة وكتابتها فقط من بين الخيارات (أ، ب، ج، د).
للإجابة، اكتب أولاً رقم الخيار (أ، ب، ج، د) ثم اكتب الإجابة. اتبع التنسيق التالي بدقة للخيار:
أ / ب / ج / د) الإجابة
'''
zero_pred = []
for i, text in enumerate(data['Question']):
    prompt = f'''السؤال الذي يجب عليك الإجابة عليه
    :السؤال:
    {text}
    الخيارات:
    a) {str(data['Option 1'].iloc[i])}
    b) {str(data['Option 2'].iloc[i])}
    c) {str(data['Option 3'].iloc[i])}
    d) {str(data['Option 4'].iloc[i])}

    الإجابة:'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature = 0.2,
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    zero_pred.append(response)

In [ ]:
pred_zero = pd.DataFrame()
pred_zero['Predicted'] = zero_pred
pred_zero['Predicted'].value_counts()

,count
Predicted,
) 12)\n\nالثقة: 95%,2
) 2),2
ب) الشرايين,1
) ب )\n\nالثقة: 95%,1
شموع\n\nالثقة: 95%,1
...,...
) العبارة الأولى صحيحة والثانية خاطئة\n\nالثابت أن الطفرات الحقيقية تحدث في الخلايا الجنسية (البويضات والحيوانات المنوية) لأنها يمكن أن تُورث إلى الأجيال القادمة. بينما تحدث الطفرات غير الحقيقية في الخلايا الجسدية ولا تُورث عادةً.,1
د) عدد الثغور قليل وطبقة الكيوتين رقيقة وعدد الشعيرات كبير\n\nالثغور القليلة تعني فقدان أقل للماء، طبقة الكيوتين الرقيقة تسمح بمرور الماء بشكل أفضل، وعدد الشعيرات الجذرية الكبير يزيد من امتصاص الماء، مما يجعل النبات أكثر دعامة فسيولوجياً. لكن يجب التنويه أن هذا ليس بالضرورة الحال دائماً، حيث تعتمد الدعامة الفسيولوجية على العديد من العوامل الأخرى أيضاً. ولكن بناءً على الخيارات المعطاة، الخيار (د) يبدو الأكثر منطقية.,1
) العدد\n\nالثقة: 95%,1


In [ ]:
pred_zero['question'] = data['Question']
pred_zero['Option 1'] = data['Option 1']
pred_zero['Option 2'] = data['Option 2']
pred_zero['Option 3'] = data['Option 3']
pred_zero['Option 4'] = data['Option 4']
pred_zero.to_excel('Fanar-QA-Biology-ZeroShot.xlsx', index = False)

In [ ]:
y_true = data['Answer Key'].values
print(classification_report(y_true, pred_zero['Predicted Normalized'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.4348    0.5556    0.4878        36
         A/C     0.0000    0.0000    0.0000         0
           B     0.4737    0.6102    0.5333        59
           C     0.5217    0.3871    0.4444        62
           D     0.5862    0.3953    0.4722        43
           c     0.0000    0.0000    0.0000         0

    accuracy                         0.4850       200
   macro avg     0.3361    0.3247    0.3230       200
weighted avg     0.5058    0.4850    0.4844       200



# Pred Few Shot

In [ ]:
content = '''لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد الإجابة الصحيحة وكتابتها فقط من بين الخيارات (أ، ب، ج، د).
للإجابة، اكتب أولاً رقم الخيار (أ، ب، ج، د) ثم اكتب الإجابة. اتبع التنسيق التالي بدقة للخيار:
أ / ب / ج / د) الإجابة

مثال 1:
السؤال:
المقاومة الكهربائية لمصباح مكتوب عليه 220 فولت ، 100 واط  هي

الخيارات:
أ) 220 أوم
ب) 202 أوم
ج) 100 أوم
د) 484 أوم

الإجابة:
د) 484 أوم

مثال 2:
السؤال:
يتشابه عملها مع عمل كرات الدم البيضاء فى الإنسان

الخيارات:
أ) جلوكوزيدات
ب) مستقبلات
ج) أحماض أمينية غير بروتينية
د) سيفالوسبورين

الإجابة:
ب) مستقبلات

مثال 3:
السؤال:
ماذا يسمى العدد 50 في عملية الطرح : 70 – 20 = 50 ؟

الخيارات:
أ) ناتج الطرح
ب) المطروح
ج) المطروح منه
د) لاشيء مما سبق

الإجابة:
أ) ناتج الطرح
'''

few_pred = []
for i, text in enumerate(data['Question']):
    prompt = f'''السؤال الذي يجب عليك الإجابة عليه
    :السؤال:
    {text}
    الخيارات:
    a) {str(data['Option 1'].iloc[i])}
    b) {str(data['Option 2'].iloc[i])}
    c) {str(data['Option 3'].iloc[i])}
    d) {str(data['Option 4'].iloc[i])}

    الإجابة:'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature = 0.2,
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    few_pred.append(response)

In [ ]:
pred_few = pd.DataFrame()
pred_few['Predicted'] = few_pred
pred_few['Predicted'].value_counts()

,count
Predicted,
) 12,2
) التمهيدي,2
) 14,2
) ATP,2
) 16\n\nفي مرحلة النمو الأول (G1)، تحتوي الخلية على عدد ثابت من الكروموسومات، وهو 16 في هذه الحالة. لا يتضاعف عدد الكروموسومات حتى مرحلة النمو الثاني (S) حيث يتم نسخ الحمض النووي.,1
...,...
) الهرمونات,1
) العباره الاولى صحيحه والثانيه خطا,1
) عدد الثغور قليل وطبقة الكيوتين سميكة وعدد الشعيرات الجذرية كبير\n\nلكن يجب التنويه أن هذا ليس بالضرورة صحيح تماماً. النباتات التي لديها عدد أقل من الثغور قد تكون أكثر مقاومة للجفاف لأنها تفقد كمية أقل من الماء عن طريق النتح. ولكن، الكيوتين السميك قد يقلل من قدرة النبات على التنفس، بينما الشعيرات الجذرية الكثيرة تساعد في امتصاص الماء والمواد المغذية. لكن، من بين الخيارات المعطاة، الخيار (ب) قد يكون الأكثر دقة فيما يتعلق بالدعم الفسيولوجي في ظروف الجفاف. لكن، يجب أن نلاحظ أن الدعم الفسيولوجي يتأثر بعدة عوامل أخرى أيضاً.,1


In [ ]:
pred_few['question'] = data['Question']
pred_few['Option 1'] = data['Option 1']
pred_few['Option 2'] = data['Option 2']
pred_few['Option 3'] = data['Option 3']
pred_few['Option 4'] = data['Option 4']
pred_few.to_excel('Fanar-QA-Biology-FewShot.xlsx', index = False)

In [ ]:
pred_few['Predicted Normalized'].value_counts()

,count
Predicted Normalized,
B,63
C,48
A,48
D,37
Unclassified,2
A/C,1
A/B,1


In [ ]:
y_true = data['Answer Key'].values
print(classification_report(y_true, pred_few['Predicted Normalized'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.3542    0.4722    0.4048        36
         A/B     0.0000    0.0000    0.0000         0
         A/C     0.0000    0.0000    0.0000         0
           B     0.4762    0.5085    0.4918        59
           C     0.5000    0.3871    0.4364        62
           D     0.5405    0.4651    0.5000        43
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.4550       200
   macro avg     0.2673    0.2618    0.2618       200
weighted avg     0.4754    0.4550    0.4607       200



# CoT

In [ ]:
content = '''لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد الإجابة الصحيحة وكتابتها فقط من بين الخيارات (أ، ب، ج، د).
للإجابة، اكتب أولاً رقم الخيار (أ، ب، ج، د) ثم اكتب الإجابة. اتبع التنسيق التالي بدقة للخيار:
أ / ب / ج / د) الإجابة

اتبع الخطوات التالية للوصول إلى الإجابة الصحيحة:

الخطوة 1: فهم السؤال
اقرأ السؤال بعناية لفهم ما يُطلب منك تحديداً. حاول فهم الحاجة والسياق الكامن خلف السؤال.

الخطوة 2: تحليل الخيارات
اقرأ كل خيار بعناية. فكّر في معنى كل خيار وكيف يرتبط بالسؤال.

الخطوة 3: تقييم الخيارات
استخدم معرفتك ومعلوماتك ذات الصلة لتقييم كل خيار. استبعد الخيارات الخاطئة بشكل واضح، ثم قارن بين الخيارات المتبقية لتحديد الأنسب.

الخطوة 4: اختيار أفضل إجابة
اختر الخيار الأكثر منطقية وصحة بناءً على التقييم في الخطوات السابقة.

الخطوة 5: كتابة الإجابة
اكتب الإجابة الصحيحة فقط باستخدام هذا التنسيق الصارم:
أ / ب / ج / د) الإجابة

مثال 1:
السؤال:
المقاومة الكهربائية لمصباح مكتوب عليه 220 فولت ، 100 واط  هي

الخيارات:
أ) 220 أوم
ب) 202 أوم
ج) 100 أوم
د) 484 أوم

التحليل:
- الخطوة 1: فهم السؤال
المطلوب حساب المقاومة الكهربائية لمصباح قدرته 100 واط وجهده 220 فولت.
- الخطوة 2: تحليل الخيارات
لحساب المقاومة نستخدم الجهد والقدرة الكهربائية المعطاة. الخيارات هي 220 أوم، 202 أوم، 100 أوم، و484 أوم.
- الخطوة 3: تقييم الخيارات
نستخدم العلاقة التالية لحساب المقاومة:
R = V^2/P
نعوض القيم المعطاة:
R = 220^2/100 = 48400/100 = 484 أوم
- الخطوة 4: اختيار الإجابة الصحيحة
من بين الخيارات، 484 أوم هي الإجابة الصحيحة، وهي الخيار د.

الإجابة النهائية:
د) 484 أوم

مثال 2:
السؤال:
يتشابه عملها مع عمل كرات الدم البيضاء فى الإنسان

الخيارات:
أ) جلوكوزيدات
ب) مستقبلات
ج) أحماض أمينية غير بروتينية
د) سيفالوسبورين

التحليل:
- الخطوة 1: فهم السؤال
السؤال يطلب: أي من الخيارات التالية يتشابه عمله مع عمل كرات الدم البيضاء في جسم الإنسان؟
كرات الدم البيضاء هي جزء من جهاز المناعة؛ وظيفتها الأساسية التعرف على مسببات الأمراض (بكتيريا، فيروسات، إلخ) ومهاجمتها.
- الخطوة 2: تحليل الخيارات
أ) جلوكوزيدات: مركبات سكرية؛ ليست جزءاً من جهاز المناعة.
ب) مستقبلات: المستقبلات تلعب دوراً مهماً في التعرف على مسببات الأمراض وإرسال إشارات لتحفيز الاستجابة المناعية.
ج) أحماض أمينية غير بروتينية: مركبات نادرة؛ ليست مرتبطة مباشرة بوظيفة جهاز المناعة.
د) سيفالوسبورين: مضاد حيوي يقضي على البكتيريا لكنه ليس مكوناً خلوياً في جهاز المناعة.
- الخطوة 3: تقييم الخيارات
ما الذي يشبه وظيفة كرات الدم البيضاء — أي القدرة على التعرف على مسببات الأمراض والاستجابة لها؟
المستقبلات تلعب دوراً رئيسياً في التعرف على الميكروبات وتفعيل الاستجابات المناعية.
- الخطوة 4: اختيار الإجابة الصحيحة
الخيار الأقرب لوظيفة كرات الدم البيضاء هو المستقبلات.

الإجابة النهائية:
ب) مستقبلات


مثال 3:
السؤال:
ماذا يسمى العدد 50 في عملية الطرح : 70 – 20 = 50 ؟

الخيارات:
أ) ناتج الطرح
ب) المطروح
ج) المطروح منه
د) لاشيء مما سبق

التحليل:
- الخطوة 1: فهم السؤال
السؤال هو: بماذا يسمى العدد 50 في عملية الطرح 70 – 20 = 50 ؟
في عملية الطرح:
العدد الأول (70) يسمى المطروح منه.
العدد الثاني (20) يسمى المطروح.
الناتج (50) يسمى ناتج الطرح أو الفرق.
- الخطوة 2: تحليل الخيارات
ناتج الطرح: هو العدد الناتج من عملية الطرح — وهو 50 في هذه الحالة.
المطروح: هو العدد الذي يتم طرحه — أي 20، وليس 50.
المطروح منه: هو العدد الذي نطرح منه — أي 70، وليس 50.
لاشيء مما سبق: غير صحيح لأن الخيار (أ) صحيح.
- الخطوة 3: تقييم الخيارات
العدد 50 هو ناتج عملية الطرح.
- الخطوة 4: اختيار الإجابة الصحيحة
الإجابة الصحيحة هي ناتج الطرح.

الإجابة النهائية:
أ) ناتج الطرح
'''

cot_pred = []
for i, text in enumerate(data['Question']):
    prompt = f'''السؤال الذي يجب عليك الإجابة عليه
    :السؤال:
    {text}
    الخيارات:
    a) {str(data['Option 1'].iloc[i])}
    b) {str(data['Option 2'].iloc[i])}
    c) {str(data['Option 3'].iloc[i])}
    d) {str(data['Option 4'].iloc[i])}'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature = 0.2,
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    cot_pred.append(response)

In [ ]:
pred_cot = pd.DataFrame()
pred_cot['Predicted'] = cot_pred
pred_cot['Predicted'].value_counts()

,count
Predicted,
) 12\n\nالتفسير:\nفي دورة كالفن (أو دورة كريبس-هالدان)، يتم استخدام 6 جزيئات NADP+ لتكوين 6 جزيئات NADPH. ومع ذلك، بما أن السؤال يتعلق بعدد جزيئات مستقبل الإلكترونات النهائي في التنفس الخلوي (وهو 18)، فإن هذا لا يرتبط مباشرة بعدد جزيئات NADPH المستخدمة في دورة كالفن. ولكن، في دورة كالفن، يتم استخدام 6 جزيئات NADP+ لتكوين 6 جزيئات NADPH. لذلك، إذا كنا ننظر فقط إلى دورة كالفن، فإن الإجابة الصحيحة هي 6. لكن بما أن الخيارات لا تتضمن 6، فإن الخيار الأقرب هو 12، وهو ضعف عدد جزيئات NADP+ المستخدمة. ومع ذلك، يجب أن نلاحظ أن السؤال يربط بين عدد جزيئات مستقبل الإلكترونات النهائي في التنفس الخلوي ودورة كالفن، وهو ما لا يوجد علاقة مباشرة بينهما. ولكن، بناءً على المعلومات المتاحة، يمكننا القول أن 6 جزيئات NADPH يتم تكوينها في دورة كالفن، ولكن الخيار الأقرب في الخيارات المعطاة هو 12. لكن يجب أن نلاحظ أن هذا ليس الإجابة الصحيحة بشكل دقيق بناءً على السؤال المعطى. إذا كنا نعتبر فقط دورة كالفن، فإن الإجابة الصحيحة هي 6. ولكن، بما أن الخيارات لا تتضمن 6، فإن الخيار الأقرب هو 12. ومع ذلك، بناءً على السؤال المعطى، لا يوجد علاقة مباشرة بين عدد جزيئات مستقبل الإلكترونات النهائي في التنفس الخلوي ودورة كالفن، وبالتالي، لا يمكننا تحديد عدد جزيئات NADPH المستخدمة في دورة كالفن بناءً على المعلومات المعطاة. لذلك، لا يوجد إجابة صحيحة من بين الخيارات المعطاة بناءً على السؤال المعطى.\n\n**لكن**، إذا كنا نعتبر فقط دورة كالفن، فإن الإجابة الصحيحة هي 6. ولكن، بما أن الخيارات لا تتضمن 6، فإن الخيار الأقرب هو 12. ومع ذلك، بناءً على السؤال المعطى، لا,1
) الخلايا وحيدة النواة\n\nالتفسير: الخلايا الوحيدة النواة (Monocytes) هي نوع من خلايا الدم البيضاء التي يمكن أن تتحول إلى خلايا بلعمية (Macrophages) عند حاجة الجسم إليها.,1
) الشرايين,1
) 16\n\nفي مرحلة النمو الأول (G1)، تحتوي الخلية على عدد ثابت من الكروموسومات، وهو 16 في هذه الحالة. لا يتغير عدد الكروموسومات خلال مرحلة النمو الأول.,1
) شموع\n\nالتفسير:\nشموع ليست من وسائل المناعة التركيبية الموجودة سلفا في النبات. فهي مادة دهنية تنتجها النباتات لحماية نفسها من فقدان الماء، لكنها ليست جزءا من آليات الدفاع ضد المهاجمين البيولوجيين مثل الشعيرات، الأشواك، والأصماغ.,1
...,...
\nالتحليل:\n\n- الخطوة 1: فهم السؤال\nالسؤال يطلب عدد جزيئات الغلوكوز التي دخلت في عملية التنفس الخلوي الهوائي بناءً على تكوين حمض الستريك 6 مرات.\n\n- الخطوة 2: تحليل الخيارات\nالخيارات هي 12، 6، 3، و 2.\n\n- الخطوة 3: تقييم الخيارات\nفي عملية التنفس الخلوي الهوائي، يتم تحويل جزيء واحد من الغلوكوز إلى 2 جزيئات من حمض الستريك (أو حمض الستريك). ومع ذلك، في السؤال، يُذكر أن حمض الستريك تم تكوينه 6 مرات، مما يعني أن 6 جزيئات من حمض الستريك تم إنتاجها.\n\n- الخطوة 4: اختيار الإجابة الصحيحة\nبما أن 6 جزيئات من حمض الستريك تم إنتاجها، وهذا يعني أن 6 جزيئات من الغلوكوز دخلت في عملية التنفس الخلوي الهوائي.\n\nالإجابة النهائية:\nب) 6,1
د) عدد الثغور قليل وطبقة الكيوتين رقيقة وعدد الشعيرات الجذرية كبير\n\nالتفسير:\nالثغور هي فتحات صغيرة في الأوراق تسمح بتبادل الغازات، بما في ذلك ثاني أكسيد الكربون اللازم للتنفس الضوئي. إذا كان عدد الثغور قليلًا، فإن النبات سيحتفظ بالماء بشكل أفضل. طبقة الكيوتين الرقيقة تسمح بمرور الماء والغازات بشكل أفضل، مما يعزز عملية التنفس الضوئي. الشعيرات الجذرية الكبيرة تساعد في امتصاص الماء والمواد المغذية من التربة، مما يزيد من الدعامة الفسيولوجية للنبات. لذلك، الخيار (د) يوفر أفضل الظروف لدعامة فسيولوجية للنبات.,1
) العدد\n\nالتفسير:\nتتكون كل من اليد والقدم من 27 عظمة، لذلك فإن العدد هو ما يتشابه فيه عظام اليد مع عظام القدم.,1


In [ ]:
pred_cot['question'] = data['Question']
pred_cot['Option 1'] = data['Option 1']
pred_cot['Option 2'] = data['Option 2']
pred_cot['Option 3'] = data['Option 3']
pred_cot['Option 4'] = data['Option 4']
pred_cot.to_excel('Fanar-QA-Biology-CoT.xlsx', index = False)

In [ ]:
pred_cot = pd.read_excel('Fanar-QA-Biology-CoT.xlsx')

In [ ]:
print(classification_report(y_true, pred_cot['Predicted Normalized'].values, digits = 4))

              precision    recall  f1-score   support

           A     0.3696    0.4722    0.4146        36
         A/C     0.0000    0.0000    0.0000         0
           B     0.4925    0.5593    0.5238        59
           C     0.5400    0.4355    0.4821        62
           D     0.5000    0.3953    0.4416        43
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.4700       200
   macro avg     0.3170    0.3104    0.3104       200
weighted avg     0.4867    0.4700    0.4736       200

